In [144]:
# Cell 1: Install Dependencies
!pip install langchain langchain-openai langchain-community faiss-cpu pandas tiktoken python-dotenv gradio

In [145]:
# Cell 2: Imports & Setup
import os
import pandas as pd

from langchain_openai import ChatOpenAI
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document


os.environ["OPENAI_API_KEY"] = "your_openai_api_key_here"

llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("LLM and Embeddings initialized successfully!")

LLM and Embeddings initialized successfully!


In [146]:
# Cell 3: Upload and Load Datasets
import pandas as pd
df = pd.read_csv("WorldCupMatches.csv")
df.head()

,Year,Datetime,Stage,Stadium,City,Home Team Name,Home Team Goals,Away Team Goals,Away Team Name,Win conditions,Attendance,Half-time Home Goals,Half-time Away Goals,Referee,Assistant 1,Assistant 2,RoundID,MatchID,Home Team Initials,Away Team Initials
0,1930.0,13 Jul 1930 - 15:00,Group 1,Pocitos,Montevideo,France,4.0,1.0,Mexico,,4444.0,3.0,0.0,LOMBARDI Domingo (URU),CRISTOPHE Henry (BEL),REGO Gilberto (BRA),201.0,1096.0,FRA,MEX
1,1930.0,13 Jul 1930 - 15:00,Group 4,Parque Central,Montevideo,USA,3.0,0.0,Belgium,,18346.0,2.0,0.0,MACIAS Jose (ARG),MATEUCCI Francisco (URU),WARNKEN Alberto (CHI),201.0,1090.0,USA,BEL
2,1930.0,14 Jul 1930 - 12:45,Group 2,Parque Central,Montevideo,Yugoslavia,2.0,1.0,Brazil,,24059.0,2.0,0.0,TEJADA Anibal (URU),VALLARINO Ricardo (URU),BALWAY Thomas (FRA),201.0,1093.0,YUG,BRA
3,1930.0,14 Jul 1930 - 14:50,Group 3,Pocitos,Montevideo,Romania,3.0,1.0,Peru,,2549.0,1.0,0.0,WARNKEN Alberto (CHI),LANGENUS Jean (BEL),MATEUCCI Francisco (URU),201.0,1098.0,ROU,PER
4,1930.0,15 Jul 1930 - 16:00,Group 1,Parque Central,Montevideo,Argentina,1.0,0.0,France,,23409.0,0.0,0.0,REGO Gilberto (BRA),SAUCEDO Ulises (BOL),RADULESCU Constantin (ROU),201.0,1085.0,ARG,FRA


In [147]:
df = pd.read_csv("WorldCupPlayers.csv")
df.head()

,RoundID,MatchID,Team Initials,Coach Name,Line-up,Shirt Number,Player Name,Position,Event
0,201,1096,FRA,CAUDRON Raoul (FRA),S,0,Alex THEPOT,GK,NaN
1,201,1096,MEX,LUQUE Juan (MEX),S,0,Oscar BONFIGLIO,GK,NaN
2,201,1096,FRA,CAUDRON Raoul (FRA),S,0,Marcel LANGILLER,NaN,G40'
3,201,1096,MEX,LUQUE Juan (MEX),S,0,Juan CARRENO,NaN,G70'
4,201,1096,FRA,CAUDRON Raoul (FRA),S,0,Ernest LIBERATI,NaN,NaN


In [148]:
df = pd.read_csv("WorldCups.csv")
df.head()

,Year,Country,Winner,Runners-Up,Third,Fourth,GoalsScored,QualifiedTeams,MatchesPlayed,Attendance
0,1930,Uruguay,Uruguay,Argentina,USA,Yugoslavia,70,13,18,590.549
1,1934,Italy,Italy,Czechoslovakia,Germany,Austria,70,16,17,363.000
2,1938,France,Italy,Hungary,Brazil,Sweden,84,15,18,375.700
3,1950,Brazil,Uruguay,Brazil,Sweden,Spain,88,13,22,1.045.246
4,1954,Switzerland,Germany FR,Hungary,Austria,Uruguay,140,16,26,768.607


In [149]:
#Cell 6 — Load All Three Datasets
def load_datasets():
    world_cups = pd.read_csv('WorldCups.csv')
    matches = pd.read_csv('WorldCupMatches.csv')
    players = pd.read_csv('WorldCupPlayers.csv')

    print(f" WorldCups.csv loaded: {len(world_cups)} rows")
    print(f" WorldCupMatches.csv loaded: {len(matches)} rows")
    print(f" WorldCupPlayers.csv loaded: {len(players)} rows")

    return world_cups, matches, players

world_cups, matches, players = load_datasets()


 WorldCups.csv loaded: 20 rows
 WorldCupMatches.csv loaded: 4572 rows
 WorldCupPlayers.csv loaded: 37784 rows


In [150]:
# Cell 7 — Clean the Matches Dataset
def clean_matches(matches):
    
    key_cols = ['Year', 'Home Team Name', 'Away Team Name', 
                'Home Team Goals', 'Away Team Goals']
    matches_clean = matches.dropna(subset=key_cols).copy()

    # Fix data types
    matches_clean['Year'] = matches_clean['Year'].astype(int)
    matches_clean['Home Team Goals'] = matches_clean['Home Team Goals'].astype(int)
    matches_clean['Away Team Goals'] = matches_clean['Away Team Goals'].astype(int)

    # Drop true duplicate rows (the CSV has some duplicated match entries)
    matches_clean = matches_clean.drop_duplicates(
        subset=['Year', 'Datetime', 'Home Team Name', 'Away Team Name']
    )

    print(f" Original matches: {len(matches)} rows")
    print(f" Cleaned matches: {len(matches_clean)} rows")
    print(f" Dropped: {len(matches) - len(matches_clean)} rows")
    return matches_clean


matches_clean = clean_matches(matches)

 Original matches: 4572 rows
 Cleaned matches: 836 rows
 Dropped: 3736 rows


In [151]:
# Cell 8 — Verify Cleaned Matches
print("Sample of clean matches:")
print(matches_clean[['Year', 'Home Team Name', 'Away Team Name', 'Home Team Goals', 'Away Team Goals']].head(10))

Sample of clean matches:
   Year Home Team Name Away Team Name  Home Team Goals  Away Team Goals
0  1930         France         Mexico                4                1
1  1930            USA        Belgium                3                0
2  1930     Yugoslavia         Brazil                2                1
3  1930        Romania           Peru                3                1
4  1930      Argentina         France                1                0
5  1930          Chile         Mexico                3                0
6  1930     Yugoslavia        Bolivia                4                0
7  1930            USA       Paraguay                3                0
8  1930        Uruguay           Peru                1                0
9  1930          Chile         France                1                0


In [152]:
# Cell 9: Build Tournament Documents
def build_tournament_docs(world_cups):
    docs = []
    for _, row in world_cups.iterrows():
        content = (
            f"Year: {row['Year']}\n"
            f"Country: {row['Country']}\n"
            f"Winner: {row['Winner']}\n"
            f"Runners-Up: {row['Runners-Up']}\n"
            f"Third: {row['Third']}\n"
            f"Fourth: {row['Fourth']}\n"
            f"Goals Scored: {row['GoalsScored']}\n"
            f"Qualified Teams: {row['QualifiedTeams']}\n"
            f"Matches Played: {row['MatchesPlayed']}\n"
            f"Attendance: {row['Attendance']}\n"
        )
        docs.append(Document(page_content=content, metadata={"type": "tournament", "year": str(row['Year'])}))

    print(f" Built {len(docs)} tournament documents")
    return docs

tournament_docs = build_tournament_docs(world_cups)

 Built 20 tournament documents


In [153]:
# Cell 10: Build Match Documents
def build_match_docs(matches_clean):
    docs = []
    for _, row in matches_clean.iterrows():
        content = (
            f"Year: {row['Year']}\n"
            f"Stage: {row['Stage']}\n"
            f"Home Team: {row['Home Team Name']}\n"
            f"Away Team: {row['Away Team Name']}\n"
            f"Home Goals: {row['Home Team Goals']}\n"
            f"Away Goals: {row['Away Team Goals']}\n"
            f"Attendance: {row['Attendance']}\n"
            f"City: {row['City']}\n"
            f"Stadium: {row['Stadium']}\n"
        )
        docs.append(Document(
            page_content=content,
            metadata={
                "type": "match",
                "year": str(row['Year']),
                "home_team": row['Home Team Name'],
                "away_team": row['Away Team Name']
            }
        ))

    print(f" Built {len(docs)} match documents")
    return docs

match_docs = build_match_docs(matches_clean)

 Built 836 match documents


In [154]:
# Cell 11: Build Team Stats Documents
def build_team_stats(matches_clean):
    docs = []
    teams = set(matches_clean['Home Team Name'].tolist() + matches_clean['Away Team Name'].tolist())

    for team in teams:
        home = matches_clean[matches_clean['Home Team Name'] == team]
        away = matches_clean[matches_clean['Away Team Name'] == team]

        total_games = len(home) + len(away)
        total_goals_scored = home['Home Team Goals'].sum() + away['Away Team Goals'].sum()
        total_goals_conceded = home['Away Team Goals'].sum() + away['Home Team Goals'].sum()

        home_wins = len(home[home['Home Team Goals'] > home['Away Team Goals']])
        away_wins = len(away[away['Away Team Goals'] > away['Home Team Goals']])
        total_wins = home_wins + away_wins

        content = (
            f"Team: {team}\n"
            f"Total Games Played: {total_games}\n"
            f"Total Wins: {total_wins}\n"
            f"Total Goals Scored: {total_goals_scored}\n"
            f"Total Goals Conceded: {total_goals_conceded}\n"
            f"Years Participated: {sorted(matches_clean[(matches_clean['Home Team Name'] == team) | (matches_clean['Away Team Name'] == team)]['Year'].unique().tolist())}\n"
        )
        docs.append(Document(
            page_content=content,
            metadata={"type": "team_stats", "team": team}
        ))

    print(f" Built {len(docs)} team stat documents")
    return docs

team_docs = build_team_stats(matches_clean)

 Built 83 team stat documents


In [155]:
# Cell 12: Build Head-to-Head Documents
def build_h2h_docs(matches_clean):
    docs = []
    matchups = {}

    for _, row in matches_clean.iterrows():
        home = row['Home Team Name']
        away = row['Away Team Name']
        key = tuple(sorted([home, away]))

        if key not in matchups:
            matchups[key] = []
        matchups[key].append(row)

    for (team1, team2), games in matchups.items():
        team1_wins = sum(1 for g in games if
            (g['Home Team Name'] == team1 and g['Home Team Goals'] > g['Away Team Goals']) or
            (g['Away Team Name'] == team1 and g['Away Team Goals'] > g['Home Team Goals']))
        team2_wins = sum(1 for g in games if
            (g['Home Team Name'] == team2 and g['Home Team Goals'] > g['Away Team Goals']) or
            (g['Away Team Name'] == team2 and g['Away Team Goals'] > g['Home Team Goals']))
        draws = len(games) - team1_wins - team2_wins

        content = (
            f"Head-to-Head: {team1} vs {team2}\n"
            f"Total Meetings: {len(games)}\n"
            f"{team1} Wins: {team1_wins}\n"
            f"{team2} Wins: {team2_wins}\n"
            f"Draws: {draws}\n"
            f"Years Played: {[g['Year'] for g in games]}\n"
        )
        docs.append(Document(
            page_content=content,
            metadata={"type": "h2h", "team1": team1, "team2": team2}
        ))

    print(f" Built {len(docs)} head-to-head documents")
    return docs

h2h_docs = build_h2h_docs(matches_clean)

 Built 578 head-to-head documents


In [156]:
# Cell 13: Combine All Documents
all_docs = tournament_docs + match_docs + team_docs + h2h_docs

print(f" Total documents: {len(all_docs)}")

 Total documents: 1517


In [157]:
# Cell 14: Split Documents
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

split_docs = text_splitter.split_documents(all_docs)

print(f" Original documents: {len(all_docs)}")
print(f" After splitting: {len(split_docs)} chunks")

 Original documents: 1517
 After splitting: 1517 chunks


In [158]:
# Cell 15: Build FAISS Vector Store
print("⏳ Building FAISS vector store... this may take a minute...")

vectorstore = FAISS.from_documents(
    documents=split_docs,
    embedding=embeddings
)

print(f" FAISS vector store built successfully!")
print(f" Total vectors indexed: {vectorstore.index.ntotal}")

⏳ Building FAISS vector store... this may take a minute...
 FAISS vector store built successfully!
 Total vectors indexed: 1517


In [159]:
# Cell 16: Save FAISS Index
vectorstore.save_local("faiss_worldcup_index")

print("✅ FAISS index saved to 'faiss_worldcup_index/' folder!")

✅ FAISS index saved to 'faiss_worldcup_index/' folder!


In [160]:
# Cell 17: Quick Test - Similarity Search
query = "Who won the 2014 World Cup?"

results = vectorstore.similarity_search(query, k=3)

print(f"🔍 Query: {query}\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(doc.page_content)
    print()

🔍 Query: Who won the 2014 World Cup?

--- Result 1 ---
Year: 2014
Country: Brazil
Winner: Germany
Runners-Up: Argentina
Third: Netherlands
Fourth: Brazil
Goals Scored: 171
Qualified Teams: 32
Matches Played: 64
Attendance: 3.386.810

--- Result 2 ---
Year: 2014
Stage: Quarter-finals
Home Team: Argentina
Away Team: Belgium
Home Goals: 1
Away Goals: 0
Attendance: 68551.0
City: Brasilia 
Stadium: Estadio Nacional

--- Result 3 ---
Year: 2014
Stage: Group E
Home Team: Switzerland
Away Team: Ecuador
Home Goals: 2
Away Goals: 1
Attendance: 68351.0
City: Brasilia 
Stadium: Estadio Nacional



In [161]:
# Cell 18: Save FAISS index files (Jupyter)
import shutil
import os


files_to_save = [
    'faiss_worldcup_index/index.faiss',
    'faiss_worldcup_index/index.pkl'
]


for file_path in files_to_save:
    filename = os.path.basename(file_path)
    shutil.copy(file_path, filename)
    print(f"Saved: {filename} → {os.getcwd()}/{filename}")

Saved: index.faiss → /Users/shilpavg/Documents/GEN_AI/untitled folder/index.faiss
Saved: index.pkl → /Users/shilpavg/Documents/GEN_AI/untitled folder/index.pkl


In [162]:
from langchain_community.embeddings import HuggingFaceEmbeddings


embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print(" HuggingFace embeddings loaded!")

 HuggingFace embeddings loaded!


In [163]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print(" Rebuilding FAISS index with HuggingFace embeddings...")

vectorstore = FAISS.from_documents(
    documents=split_docs,
    embedding=embeddings
)

print(f" FAISS index rebuilt!")
print(f" Total vectors: {vectorstore.index.ntotal}")

 Rebuilding FAISS index with HuggingFace embeddings...
 FAISS index rebuilt!
 Total vectors: 1517


In [164]:
vectorstore.save_local("faiss_worldcup_index_hf")
print(" Saved!")

 Saved!


In [165]:
import shutil
import os

output_dir = './faiss_worldcup_index_hf_output'
os.makedirs(output_dir, exist_ok=True)

files_to_save = [
    'faiss_worldcup_index_hf/index.faiss',
    'faiss_worldcup_index_hf/index.pkl'
]

for file_path in files_to_save:
    filename = os.path.basename(file_path)
    dest = os.path.join(output_dir, filename)
    shutil.copy(file_path, dest)
    print(f"Saved: {dest}")

Saved: ./faiss_worldcup_index_hf_output/index.faiss
Saved: ./faiss_worldcup_index_hf_output/index.pkl


In [166]:
!pip install nbformat
import nbformat

with open('WorldCup_Final.ipynb', 'r') as f:
    nb = nbformat.read(f, as_version=4)


if 'widgets' in nb.metadata:
    del nb.metadata['widgets']


with open('WorldCup_Final.ipynb', 'w') as f:
    nbformat.write(nb, f)

print(" Notebook cleaned!")

 Notebook cleaned!


In [167]:
import nbformat
import json


with open('WorldCup_Final.ipynb', 'r') as f:
    nb = json.load(f)

if 'widgets' in nb.get('metadata', {}):
    del nb['metadata']['widgets']

for cell in nb.get('cells', []):
    if 'metadata' in cell:
        if 'widgets' in cell['metadata']:
            del cell['metadata']['widgets']
    # Clear outputs that contain widget data
    if 'outputs' in cell:
        new_outputs = []
        for output in cell['outputs']:
            if output.get('output_type') != 'display_data':
                new_outputs.append(output)
        cell['outputs'] = new_outputs

with open('WorldCup_Final.ipynb', 'w') as f:
    json.dump(nb, f)

print(" Notebook fully cleaned!")

 Notebook fully cleaned!


In [168]:
import nbformat


with open('WorldCup_Final.ipynb', 'r') as f:
    nb = nbformat.read(f, as_version=4)


for cell in nb.cells:
    if hasattr(cell, 'outputs'):
        cell.outputs = []
    if hasattr(cell, 'execution_count'):
        cell.execution_count = None
    cell.metadata = {}


nb.metadata.pop('widgets', None)


with open('WorldCup_Final.ipynb', 'w') as f:
    nbformat.write(nb, f)

print("All outputs cleared!")

All outputs cleared!


In [169]:
import os
import pandas as pd
from langchain_ollama import ChatOllama
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.tools import tool
from langchain_classic.agents import AgentExecutor, create_tool_calling_agent

# LLM - Ollama 
llm = ChatOllama(model="llama3.2", temperature=0.3)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


store = {}

def get_session_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    
    history = store[session_id]
    if len(history.messages) > 10:
        history.messages = history.messages[-10:]
    return history

print(" Setup complete")

 Setup complete


In [170]:
# vectorstore = FAISS.load_local("faiss_worldcup_index", embeddings, allow_dangerous_deserialization=True)

# vectorstore = FAISS.from_documents(mock_docs, embeddings)
# print(f" Mock vectorstore ready with {len(mock_docs)} documents")

from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings


embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = FAISS.load_local(
    "faiss_worldcup_index_hf",
    embeddings,
    allow_dangerous_deserialization=True
)

print(f" Vectorstore loaded with {vectorstore.index.ntotal} vectors")

 Vectorstore loaded with 1517 vectors


In [171]:
# Create a Retriever and test it

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

results = retriever.invoke("Who won the 2014 World Cup?")

print(f"Search returned {len(results)} documents:\n")
for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(doc.page_content[:150])
    print()

Search returned 4 documents:

--- Result 1 ---
Year: 2014
Country: Brazil
Winner: Germany
Runners-Up: Argentina
Third: Netherlands
Fourth: Brazil
Goals Scored: 171
Qualified Teams: 32
Matches Playe

--- Result 2 ---
Year: 2014
Stage: Quarter-finals
Home Team: Argentina
Away Team: Belgium
Home Goals: 1
Away Goals: 0
Attendance: 68551.0
City: Brasilia 
Stadium: Esta

--- Result 3 ---
Year: 2014
Stage: Group E
Home Team: Switzerland
Away Team: Ecuador
Home Goals: 2
Away Goals: 1
Attendance: 68351.0
City: Brasilia 
Stadium: Estadio N

--- Result 4 ---
Year: 2014
Stage: Group A
Home Team: Cameroon
Away Team: Brazil
Home Goals: 1
Away Goals: 4
Attendance: 69112.0
City: Brasilia 
Stadium: Estadio Nacio



In [172]:
# UPGRADED RAG Chain — Chain-of-Thought + Source Citations

QA_SYSTEM_PROMPT = """You are the World Cup AI Analyst — an expert football historian 
covering FIFA World Cup tournaments from 1930 to 2014.

Use ONLY the provided context to answer. Follow this process:

STEP 1 - THINK: Identify which pieces of context are relevant to the question.
STEP 2 - REASON: Connect the facts logically to form your answer.
STEP 3 - ANSWER: Give your final response with citations.

Format your response like this:

💭 Reasoning:
[1-2 sentences showing your thought process — which data you're using and why]

📋 Answer:
[Your clear, concise answer — lead with the direct answer, then supporting details]
[Use specific numbers: years, scores, goals, attendance]
[Keep it 3-5 sentences for simple questions, more for complex ones]

📎 Sources:
[List which data you used, e.g., "Tournament data: 2014 World Cup", "Team stats: Brazil"]

Rules:
- NEVER make up statistics. Only use what's in the context.
- If the context doesn't have enough info, say: "Based on available data..." and state what's missing.
- Be engaging — like a football commentator who loves the game.
- This data covers 1930-2014 only. Say so if asked about other years.

Context:
{context}
"""

qa_prompt = ChatPromptTemplate.from_messages([
    ("system", QA_SYSTEM_PROMPT),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}")
])

question_answer_chain = create_stuff_documents_chain(llm, qa_prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

print("Upgraded RAG chain — chain-of-thought + citations")

Upgraded RAG chain — chain-of-thought + citations


In [173]:
# Add Memory + Ask Function

memory = ConversationBufferWindowMemory(
    k=5,
    memory_key="chat_history",
    return_messages=True
)

def ask_worldcup(question: str) -> str:
    """Ask a question about World Cup history with memory."""
    chat_history = memory.load_memory_variables({})["chat_history"]
    
    response = rag_chain.invoke({
        "input": question,
        "chat_history": chat_history
    })
    
    memory.save_context(
        {"input": question},
        {"output": response["answer"]}
    )
    
    return response["answer"]

print(" Memory + ask function ready")

 Memory + ask function ready


In [174]:
#Test the RAG Chain

print("Q1: Who won the 2014 World Cup?")
print("A:", ask_worldcup("Who won the 2014 World Cup?"))
print()
print("=" * 50)
print()
print("Q2: How many goals were scored in that tournament?")
print("A:", ask_worldcup("How many goals were scored in that tournament?"))

Q1: Who won the 2014 World Cup?
A: 🏆 Germany won the 2014 FIFA World Cup! They defeated Argentina 1-0 in the final on July 13, 2014, at the Maracanã Stadium in Rio de Janeiro, Brazil. This was their fourth World Cup title.

📋 Answer:
The winner of the 2014 World Cup was Germany.
They won the tournament by defeating Argentina 1-0 in the final match.
This marked Germany's fourth World Cup victory.

📎 Sources:
"Tournament data: 2014 World Cup"
"Final Match Result: Germany vs. Argentina (July 13, 2014)"

Note: The context doesn't provide information on the exact score of the final match or other details about the tournament.


Q2: How many goals were scored in that tournament?
A: 📊 According to the available data, a total of 171 goals were scored during the 2014 World Cup.

📋 Answer:
A total of 171 goals were scored during the 2014 FIFA World Cup.
This is based on the available data from the tournament.

📎 Sources:
"Tournament data: 2014 World Cup"


In [175]:
#Prediction chain built

PREDICTION_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are an elite World Cup match analyst. Using historical data provided,
produce a data-driven match preview using the STAR analysis framework.

Use the following context:
{context}

Structure your response EXACTLY like this:

🏟️ MATCH ANALYSIS: [Team1] vs [Team2]

📌 SITUATION
- [Team1]: [W]-[D]-[L] record, [win rate]%, [titles] title(s) in [years]
- [Team2]: [W]-[D]-[L] record, [win rate]%, [titles] title(s) in [years]
- Head-to-head: [X] meetings — [summary of who leads]
- Last meeting: [match and score]

🎯 TASK
- [Team1] must: [what data says they need to do — e.g., "overcome a 1-4 H2H deficit"]
- [Team2] must: [what data says they need to do]

⚡ ACTION
- [Team1] strength: [backed by specific stat from context]
- [Team2] strength: [backed by specific stat from context]
- Deciding factor: [one key stat that tips the balance]

🏆 RESULT
- Prediction: [Team1] [X] - [Y] [Team2]
- Winner: [team]
- Reasoning: [2-3 sentences citing specific numbers from the data above]
- Confidence: [Low/Medium/High] — [why]

📎 Data Used: [List sources — e.g., "Brazil team stats", "Brazil vs Germany H2H record", "2014 Semi-final match data"]

⚠️ This analysis uses historical FIFA World Cup data (1930-2014) for educational purposes only.
It does not reflect current form, rosters, or real-world conditions."""),
    ("human", "Predict the match outcome: {team1} vs {team2}")
])

prediction_chain = create_stuff_documents_chain(llm, PREDICTION_PROMPT)

print("Prediction chain built")

Prediction chain built


In [176]:
# Predict Match Function

def predict_match(team1: str, team2: str) -> str:
    """Generate a match prediction between two teams."""
    
    
    h2h_docs = vectorstore.similarity_search(
        f"{team1} vs {team2} head to head World Cup", k=3
    )
    
    
    team1_docs = vectorstore.similarity_search(
        f"{team1} World Cup statistics record", k=2
    )
    
    
    team2_docs = vectorstore.similarity_search(
        f"{team2} World Cup statistics record", k=2
    )
    
   
    all_docs = h2h_docs + team1_docs + team2_docs
    seen = set()
    unique_docs = []
    for doc in all_docs:
        if doc.page_content not in seen:
            seen.add(doc.page_content)
            unique_docs.append(doc)
    
    print(f"   Retrieved {len(unique_docs)} unique documents for prediction")
    
   
    response = prediction_chain.invoke({
        "context": unique_docs,
        "team1": team1,
        "team2": team2
    })
    
    return response

print("Predict function ready")

Predict function ready


In [177]:
# Test Prediction

print("MATCH PREDICTION: Brazil vs Germany")
print(predict_match("Brazil", "Germany"))

MATCH PREDICTION: Brazil vs Germany
   Retrieved 7 unique documents for prediction
🏟️ MATCH ANALYSIS: Brazil vs Germany

📌 SITUATION
- Brazil: 70W-10D-24L, 67% win rate, 0 titles in 2014
- Germany: 30W-6D-8L, 64% win rate, 1 title in 1994
- Head-to-head: 2 meetings — Brazil leads 1-1
- Last meeting: 2014 Semi-final match (Germany won 7-1)

🎯 TASK
- Brazil must overcome a 1-4 H2H deficit and capitalize on Germany's recent struggles.
- Germany must improve their performance against top-tier teams like Brazil.

⚡ ACTION
- Brazil strength: 221 goals scored in 104 games, averaging 2.12 goals per match.
- Germany strength: 93 goals conceded in 44 games, allowing an average of 2.11 goals per match.
- Deciding factor: Brazil's goal-scoring ability vs. Germany's defensive solidity.

🏆 RESULT
- Prediction: Brazil +1/2 - 3/4 Germany
- Winner: Brazil
- Reasoning: Brazil's impressive goal-scoring record and recent form make them the favorites, while Germany's struggles against top teams and defensi

In [178]:
# Define Agent Tools
user_preferences = {}

@tool
def dataset_discovery_tool(query: str) -> str:
    """Discover what data is available in the World Cup dataset.
    Use this when the user asks what data you have, what years are covered,
    or what teams/tournaments are in the dataset."""
    return (
        "The dataset covers 20 FIFA World Cup tournaments from 1930 to 2014.\n"
        "Total matches: 852\n"
        "Total teams: 83\n"
        "Data includes: tournament summaries, match-level results, team statistics, "
        "head-to-head records, and player participation records.\n"
        "Teams include: Brazil, Germany, Argentina, France, Italy, Spain, "
        "England, Netherlands, Uruguay, and 74 more."
    )

@tool
def data_ingestion_tool(team_name: str) -> str:
    """Look up raw statistical data for a specific team from the dataset.
    Returns match counts, goals, and tournament participation.
    Use this when the user asks for a specific team's stats or record."""
    docs = vectorstore.similarity_search(f"{team_name} World Cup statistics record", k=2)
    if docs:
        if team_name.lower() in [v.lower() for v in user_preferences.values()]:
            prefix = f"Here are the stats for your favorite team, {team_name}!\n\n"
        else:
            prefix = ""
        return prefix + docs[0].page_content
    return f"No data found for '{team_name}'. Check the team name spelling."

@tool
def retrieval_or_filter_tool(question: str) -> str:
    """Search the World Cup knowledge base to answer factual questions about
    World Cup history, tournament results, match outcomes, and team records.
    Use this for any question about World Cup facts and history."""
    docs = vectorstore.similarity_search(question, k=6)
    return "\n\n".join([d.page_content for d in docs])

@tool
def reasoning_or_aggregation_tool(matchup: str) -> str:
    """Generate a match prediction between two teams. Input should be in
    the format 'Team1 vs Team2', e.g., 'Brazil vs Germany'.
    Analyzes head-to-head records and team stats to predict outcomes."""
    parts = matchup.split(" vs ")
    if len(parts) != 2:
        return "Please provide the matchup in format: 'Team1 vs Team2'"
    return predict_match(parts[0].strip(), parts[1].strip())

@tool
def report_generation_tool(topic: str) -> str:
    """Generate a structured report or summary about a World Cup topic.
    Good for questions like 'summarize the 1998 World Cup' or
    'give me a report on Brazil's World Cup history'."""
    docs = vectorstore.similarity_search(topic, k=8)
    context = "\n\n".join([d.page_content for d in docs])
    
    report_prompt = ChatPromptTemplate.from_messages([
        ("system", "Generate a well-structured report on the given topic using the context. "
         "Include relevant statistics, match results, and historical context. "
         "Note this covers World Cups 1930-2014.\n\nContext:\n{context}"),
        ("human", "{topic}")
    ])
    chain = report_prompt | llm
    return chain.invoke({"context": context, "topic": topic}).content

@tool
def comparison_tool(matchup: str) -> str:
    """Compare two teams side-by-side with detailed statistics.
    Input format: 'Team1 vs Team2', e.g., 'Brazil vs Argentina'.
    Use this when the user asks to compare teams, asks which team is better,
    or wants a side-by-side analysis."""
    parts = matchup.split(" vs ")
    if len(parts) != 2:
        return "Please provide in format: 'Team1 vs Team2'"
    
    team1, team2 = parts[0].strip(), parts[1].strip()
    
    t1_docs = vectorstore.similarity_search(f"{team1} World Cup statistics", k=2)
    t2_docs = vectorstore.similarity_search(f"{team2} World Cup statistics", k=2)
    h2h_docs = vectorstore.similarity_search(f"{team1} vs {team2} head to head", k=2)
    
    all_context = t1_docs + t2_docs + h2h_docs
    context = "\n\n".join([d.page_content for d in all_context])
    
    compare_prompt = ChatPromptTemplate.from_messages([
        ("system", """Compare these two World Cup teams side-by-side using the data provided.

Format your response as:

📊 TEAM COMPARISON: [Team1] vs [Team2]

| Category | [Team1] | [Team2] | Edge |
|----------|---------|---------|------|
| Matches Played | [X] | [Y] | [who leads] |
| Win Rate | [X%] | [Y%] | [who leads] |
| Goals Scored | [X] | [Y] | [who leads] |
| Goals Conceded | [X] | [Y] | [who leads] |
| World Cup Titles | [X] | [Y] | [who leads] |

⚔️ HEAD-TO-HEAD: [summary]

🏆 VERDICT: [Which team has the statistical edge overall and why — 2-3 sentences]

📎 Data Used: [list sources]

Context:
{context}"""),
        ("human", "Compare {team1} vs {team2}")
    ])

    chain = compare_prompt | llm
    return chain.invoke({"context": context, "team1": team1, "team2": team2}).content

@tool
def set_user_preference(preference: str) -> str:
    """Save a user preference like their favorite team.
    Input format: 'favorite_team: Brazil' or 'favorite_team: Germany'.
    Use this when the user says 'my favorite team is...' or 'I support...'."""
    if ":" in preference:
        key, value = preference.split(":", 1)
        user_preferences[key.strip()] = value.strip()
        return f"Got it! I've noted your {key.strip()} as {value.strip()}. I'll personalize responses accordingly."
    return "Please provide in format: 'favorite_team: Brazil'"

tools = [
    dataset_discovery_tool,
    data_ingestion_tool,
    retrieval_or_filter_tool,
    reasoning_or_aggregation_tool,
    report_generation_tool,
    comparison_tool,
    set_user_preference
]

print(f" {len(tools)} tools defined")

 7 tools defined


In [179]:
#Build the Agent 

AGENT_SYSTEM_PROMPT = """You are the World Cup AI Analyst, an expert chatbot on FIFA World Cup 
history from 1930 to 2014.

You have 6 tools:
- dataset_discovery_tool: Find what data is available
- data_ingestion_tool: Look up raw stats for a specific team
- retrieval_or_filter_tool: Search the knowledge base for factual answers
- reasoning_or_aggregation_tool: Predict match outcomes (format: 'Team1 vs Team2')
- report_generation_tool: Generate structured reports on topics
- comparison_tool: Compare two teams side-by-side (format: 'Team1 vs Team2')

TOOL SELECTION GUIDE:
- "Who won..." / "How many..." / "When did..." → retrieval_or_filter_tool
- "Predict..." / "What would happen if..." → reasoning_or_aggregation_tool
- "Compare..." / "Which is better..." / "X vs Y stats" → comparison_tool
- "Tell me about [team]..." / "[team] stats" → data_ingestion_tool
- "Summarize..." / "Report on..." → report_generation_tool
- "What data..." / "What teams..." → dataset_discovery_tool

USER PREFERENCE HANDLING:
- If the user says "my favorite team is X" or "I support X", remember it and use it 
  to personalize future responses.
- If the user has a favorite team, relate answers back to their team when relevant.

Guidelines:
- Always use tools to ground responses in real data.
- Be enthusiastic and conversational about football.
- Cite specific stats, scores, and years.
- For predictions, note they are educational and based on historical data only.
- If data is insufficient, say so honestly.
"""

agent_prompt = ChatPromptTemplate.from_messages([
    ("system", AGENT_SYSTEM_PROMPT),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

agent = create_tool_calling_agent(llm, tools, agent_prompt)

agent_memory = ConversationBufferWindowMemory(
    k=5,
    memory_key="chat_history",
    return_messages=True
)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    memory=agent_memory,
    verbose=True,
    handle_parsing_errors=True,
    max_iterations=5
)

print("Agent ready ")

Agent ready 


In [180]:
#Test the Agent

print("TEST 1: Factual question")
response = agent_executor.invoke({"input": "Who won the 2014 World Cup?"})
print("\nFinal Answer:", response["output"])

TEST 1: Factual question


> Entering new AgentExecutor chain...

Invoking: `retrieval_or_filter_tool` with `{'question': 'Who won the 2014 World Cup?'}`


Year: 2014
Country: Brazil
Winner: Germany
Runners-Up: Argentina
Third: Netherlands
Fourth: Brazil
Goals Scored: 171
Qualified Teams: 32
Matches Played: 64
Attendance: 3.386.810

Year: 2014
Stage: Quarter-finals
Home Team: Argentina
Away Team: Belgium
Home Goals: 1
Away Goals: 0
Attendance: 68551.0
City: Brasilia 
Stadium: Estadio Nacional

Year: 2014
Stage: Group E
Home Team: Switzerland
Away Team: Ecuador
Home Goals: 2
Away Goals: 1
Attendance: 68351.0
City: Brasilia 
Stadium: Estadio Nacional

Year: 2014
Stage: Group A
Home Team: Cameroon
Away Team: Brazil
Home Goals: 1
Away Goals: 4
Attendance: 69112.0
City: Brasilia 
Stadium: Estadio Nacional

Year: 2014
Stage: Final
Home Team: Germany
Away Team: Argentina
Home Goals: 1
Away Goals: 0
Attendance: 74738.0
City: Rio De Janeiro 
Stadium: Estadio do Maracana

Year: 2014
Stage: Quart

In [181]:
#Test prediction and memory

print("TEST 2: Match Prediction")
response = agent_executor.invoke({"input": "Predict Brazil vs Germany"})
print("\nFinal Answer:", response["output"])

TEST 2: Match Prediction


> Entering new AgentExecutor chain...

Invoking: `reasoning_or_aggregation_tool` with `{'matchup': 'Brazil vs Germany'}`


   Retrieved 7 unique documents for prediction
🏟️ MATCH ANALYSIS: Brazil vs Germany

📌 SITUATION
- Brazil: 70-30-44 record, 69% win rate, 5 World Cup titles in [1930, 1958, 1962, 1970, 1994, 2002]
- Germany: 30-44-62 record, 40% win rate, 4 World Cup titles in [1954, 1974, 1990, 2014]
- Head-to-head: 1 meeting — Brazil won
- Last meeting: 2014 Semi-final match, Brazil won 7-1

🎯 TASK
- Brazil must overcome a significant H2H deficit and rely on their high-scoring offense to outdo Germany's defensive solidity.
- Germany must capitalize on Brazil's vulnerability in big games and exploit their tendency to concede goals.

⚡ ACTION
- Brazil strength: High Goals Scored (221) against weaker opponents, indicating a potent attacking force.
- Germany strength: Low Goals Conceded (44) in World Cup matches, suggesting a solid defensive foundation.
- D

In [182]:
# Test memory

print("TEST 3: Memory test")
response = agent_executor.invoke({"input": "How many goals were scored in that tournament we discussed?"})
print("\nFinal Answer:", response["output"])

TEST 3: Memory test


> Entering new AgentExecutor chain...

Invoking: `data_ingestion_tool` with `{'team_name': 'Brazil'}`


Team: Brazil
Total Games Played: 104
Total Wins: 70
Total Goals Scored: 221
Total Goals Conceded: 102
Years Participated: [1930, 1934, 1938, 1950, 1954, 1958, 1962, 1966, 1970, 1974, 1978, 1982, 1986, 1990, 1994, 1998, 2002, 2006, 2010, 2014]According to the data ingestion tool output, Brazil scored a total of 221 goals in all their World Cup appearances from 1930 to 2014.

> Finished chain.

Final Answer: According to the data ingestion tool output, Brazil scored a total of 221 goals in all their World Cup appearances from 1930 to 2014.


In [183]:
#Test report generation

print("TEST 4: Report generation")
response = agent_executor.invoke({"input": "Give me a report on the 1998 World Cup"})
print("\nFinal Answer:", response["output"])

TEST 4: Report generation


> Entering new AgentExecutor chain...

Invoking: `report_generation_tool` with `{'topic': 'the 1998 World Cup'}`


**The 1998 FIFA World Cup: A French Triumph**

The 1998 FIFA World Cup, held in France from June 10 to July 12, 1998, was a highly anticipated event that showcased the world's best footballers. The tournament featured 32 national teams competing in 64 matches, with a total attendance of over 2.7 million spectators.

**Historical Context**

The 1998 World Cup was the 15th edition of the FIFA World Cup, and it marked the first time that France hosted the tournament. The event was also notable for being the last World Cup to be held before the introduction of the "Golden Generation" concept, where teams were expected to produce a talented group of players who would dominate international football for years to come.

**Group Stage**

The 1998 World Cup was divided into eight groups of four teams each. The top two teams from each group advanced to th

In [184]:
import gradio as gr
import re

TITLE = "⚽ World Cup AI Analyst"
DESCRIPTION = """
Ask about FIFA World Cup history (1930–2014).
If you ask for a report (e.g., "Generate a report on 2002 World Cup"),
I will format it as a clean, visually appealing report directly in the chat.
"""

EXAMPLES = [
    "Generate a report on 2002 World Cup",
    "Who won the 2014 World Cup and what was the final score?",
    "Compare Brazil vs Germany in World Cup history.",
    "Tell me France's World Cup performance from 1998 to 2014.",
    "Predict Brazil vs Germany (World Cup context)."
]

def clean_agent_output(text: str) -> str:
    """
    Removes extra narration like 'I've generated...' so chat shows only content.
    """
    if not isinstance(text, str):
        return str(text)

    # Cut off common narration that starts after the report
    text = re.split(r"\bI['’]ve generated\b", text, flags=re.IGNORECASE)[0].strip()
    text = re.split(r"\bLet me know\b", text, flags=re.IGNORECASE)[0].strip()
    text = re.split(r"\bWould you like\b", text, flags=re.IGNORECASE)[0].strip()
    return text.strip()

def looks_like_report(text: str) -> bool:
    if not isinstance(text, str):
        return False
    return (
        "World Cup Report" in text
        or ("**Introduction**" in text and "**Legacy**" in text)
        or ("Introduction" in text and "Legacy" in text)
    )

def extract_stat(pattern, text, cast=str, default="—"):
    m = re.search(pattern, text, flags=re.IGNORECASE)
    if not m:
        return default
    try:
        return cast(m.group(1))
    except:
        return m.group(1)

def pretty_report_md(raw: str) -> str:
    """
    Convert the report-like text into a nicer Markdown layout.
    Works even if the output is already markdown.
    """
    txt = raw.strip()

    # Title (try to pull something like "**2002 FIFA World Cup Report**")
    title = extract_stat(r"\*\*(.+?World Cup Report)\*\*", txt, str, "World Cup Report")

    # Pull a few key stats if present
    matches = extract_stat(r"Total matches played:\s*\*?\s*([0-9]+)", txt)
    goals = extract_stat(r"Total goals scored:\s*\*?\s*([0-9]+)", txt)
    avg_goals = extract_stat(r"Average goals per match:\s*\*?\s*([0-9.]+)", txt)

    winner = extract_stat(r"Winner:\s*\*?\s*([A-Za-z ]+)", txt)
    runner = extract_stat(r"Runner[- ]up:\s*\*?\s*([A-Za-z ]+)", txt)
    third = extract_stat(r"Third place:\s*\*?\s*([A-Za-z ]+)", txt)
    fourth = extract_stat(r"Fourth place:\s*\*?\s*([A-Za-z ]+)", txt)

    final_line = extract_stat(r"final match saw\s+([A-Za-z ]+)\s+defeat\s+([A-Za-z ]+)\s+([0-9]+)\s*-\s*([0-9]+)", txt, str, "")
    if final_line and final_line != "—":
        # If regex returned a combined group (it won't here). We'll do a safer approach:
        m = re.search(r"final match saw\s+([A-Za-z ]+)\s+defeat\s+([A-Za-z ]+)\s+([0-9]+)\s*-\s*([0-9]+)", txt, flags=re.IGNORECASE)
        if m:
            final_pretty = f"🏆 **Final:** **{m.group(1).strip()} {m.group(3)}–{m.group(4)} {m.group(2).strip()}**"
        else:
            final_pretty = "🏆 **Final:** —"
    else:
        final_pretty = "🏆 **Final:** —"

    # Try to extract the main body sections if present
    # We'll keep the original content below, but present a nice summary block on top.
    summary_block = f"""
### {title}

{final_pretty}

---

#### 📌 Quick Stats
- 🗓️ **Matches:** {matches}
- ⚽ **Goals:** {goals}
- 📈 **Avg goals/match:** {avg_goals}

#### 🥇 Final Standings
- 🥇 **Winner:** {winner}
- 🥈 **Runner-up:** {runner}
- 🥉 **Third:** {third}
- 4️⃣ **Fourth:** {fourth}

---

#### 📝 Full Report
{txt}
""".strip()

    return summary_block

def respond(message, messages):
    messages = messages or []
    messages.append({"role": "user", "content": message})

    if "agent_executor" not in globals():
        messages.append({"role": "assistant", "content": "⚠️ agent_executor is not defined. Run Member-2 agent build cell first."})
        return messages

    try:
        result = agent_executor.invoke({"input": message})
        raw_out = result.get("output", str(result)) if isinstance(result, dict) else str(result)

        cleaned = clean_agent_output(raw_out)

        # If it's a report, format nicely
        if looks_like_report(cleaned):
            cleaned = pretty_report_md(cleaned)

        messages.append({"role": "assistant", "content": cleaned})
        return messages

    except Exception as e:
        messages.append({"role": "assistant", "content": f"⚠️ Error: {type(e).__name__}: {e}"})
        return messages

with gr.Blocks() as demo:
    gr.Markdown(f"# {TITLE}")
    gr.Markdown(DESCRIPTION)

    chatbot = gr.Chatbot(height=460)
    msg = gr.Textbox(placeholder="Ask a World Cup question…", label="Message")

    with gr.Row():
        send = gr.Button("Send")
        clear = gr.Button("Clear")

    gr.Examples(EXAMPLES, inputs=msg)

    send.click(respond, inputs=[msg, chatbot], outputs=[chatbot])
    msg.submit(respond, inputs=[msg, chatbot], outputs=[chatbot])

    def _clear():
        return []
    clear.click(_clear, outputs=[chatbot])

demo.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.




> Entering new AgentExecutor chain...

Invoking: `retrieval_or_filter_tool` with `{'question': 'Who won the 2014 World Cup and what was the final score?'}`


Year: 2014
Country: Brazil
Winner: Germany
Runners-Up: Argentina
Third: Netherlands
Fourth: Brazil
Goals Scored: 171
Qualified Teams: 32
Matches Played: 64
Attendance: 3.386.810

Year: 2014
Stage: Quarter-finals
Home Team: Argentina
Away Team: Belgium
Home Goals: 1
Away Goals: 0
Attendance: 68551.0
City: Brasilia 
Stadium: Estadio Nacional

Year: 2014
Stage: Final
Home Team: Germany
Away Team: Argentina
Home Goals: 1
Away Goals: 0
Attendance: 74738.0
City: Rio De Janeiro 
Stadium: Estadio do Maracana

Year: 2014
Stage: Group A
Home Team: Cameroon
Away Team: Brazil
Home Goals: 1
Away Goals: 4
Attendance: 69112.0
City: Brasilia 
Stadium: Estadio Nacional

Year: 2014
Stage: Quarter-finals
Home Team: France
Away Team: Germany
Home Goals: 0
Away Goals: 1
Attendance: 74240.0
City: Rio De Janeiro 
Stadium: Estadio do Maracana

Year: 20